In [ ]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"lfreedom2750","key":"fb46b035b65134128288a6ce5f370912"}'}

In [ ]:
!pip install -q kaggle

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/

!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d lfreedom2750/fakeface-train-data-v2
!unzip -q fakeface-train-data-v2.zip -d train_dataset

Dataset URL: https://www.kaggle.com/datasets/lfreedom2750/fakeface-train-data-v2
License(s): unknown
fakeface-train-data-v2.zip: Skipping, found more recently modified local copy (use --force to force download)


In [ ]:
!kaggle datasets download -d lfreedom2750/fakeface-valid-data-v2
!unzip -q fakeface-valid-data-v2.zip -d valid_dataset

Dataset URL: https://www.kaggle.com/datasets/lfreedom2750/fakeface-valid-data-v2
License(s): unknown
 87% 648M/745M [00:00<00:00, 1.32GB/s]
100% 745M/745M [00:00<00:00, 1.21GB/s]


In [ ]:
!kaggle datasets download -d lfreedom2750/fakeface-test-data-v2
!unzip -q fakeface-test-data-v2.zip -d test_dataset

Dataset URL: https://www.kaggle.com/datasets/lfreedom2750/fakeface-test-data-v2
License(s): unknown
 87% 652M/747M [00:00<00:00, 1.27GB/s]
100% 747M/747M [00:00<00:00, 1.21GB/s]


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import csv
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score
import matplotlib.pyplot as plt
from tqdm import tqdm
import os
from PIL import Image

In [ ]:
from torchvision import datasets
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

train_dataset = datasets.ImageFolder("/content/train_dataset", transform=transform)
val_dataset   = datasets.ImageFolder("/content/valid_dataset", transform=transform)
test_dataset = datasets.ImageFolder("/content/test_dataset", transform=transform)

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True, num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=256, shuffle=False, num_workers=4, pin_memory=True)
test_loader   = DataLoader(test_dataset, batch_size=256, shuffle=False,num_workers=4, pin_memory=True)

In [ ]:
from torch.nn import functional as F
import torch
import torch.nn as nn
from torchvision import models

class ResNet50(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        num_features = self.model.fc.in_features
        self.model.fc = nn.Linear(num_features, 2)

    def forward(self, x):
        return self.model(x)

In [ ]:
import torch.nn as nn
from torchvision import models

class ResNet18(nn.Module):
    def __init__(self):
        super(ResNet18, self).__init__()
        self.model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

        num_features = self.model.fc.in_features
        self.model.fc = nn.Linear(num_features, 2)

    def forward(self, x):
        return self.model(x)

In [ ]:
import time
import torch
import torch.nn as nn
from torchvision import models
from tqdm import tqdm
from sklearn.metrics import roc_auc_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

teacher = ResNet50()
teacher = teacher.to(device)

optimizer = torch.optim.Adam(teacher.parameters(), lr=5e-5)
criterion = nn.CrossEntropyLoss()

epoch_times = []
start_training = time.time()

for epoch in range(10):
    start_epoch = time.time()

    teacher.train()
    total_loss = 0
    total_batches = len(train_loader)

    for x, y in tqdm(train_loader, desc=f"[Epoch {epoch+1}] Training", leave=False):
        x, y = x.to(device), y.to(device)
        loss = criterion(teacher(x), y)
        print(f"Loss: {loss.item():.4f}")
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        total_loss += loss.item()

    avg = total_loss / len(train_loader)

    teacher.eval()
    correct, total = 0, 0
    all_labels, all_probs = [], []

    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            outputs = teacher(x)
            probs = torch.softmax(outputs, dim=1)[:, 1]

            preds = torch.argmax(outputs, dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

            all_labels.extend(y.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    val_acc = correct / total
    val_auc = roc_auc_score(all_labels, all_probs)

    epoch_time = time.time() - start_epoch
    epoch_times.append(epoch_time)

    print(f"[Teacher] Epoch {epoch+1} | Train Loss: {avg:.4f} | Val Acc: {val_acc:.4f} | Val AUC: {val_auc:.4f} | Time: {epoch_time:.2f}s")

total_time = time.time() - start_training
avg_time = sum(epoch_times) / len(epoch_times)

print(f"\nTotal training time: {total_time:.2f}s")
print(f"Average time per epoch: {avg_time:.2f}s")

torch.save(teacher.state_dict(), "restnet50.pth")

cuda


[Epoch 1] Training:   0%|          | 0/473 [00:00<?, ?it/s]

Loss: 0.7382


[Epoch 1] Training:   0%|          | 1/473 [00:04<32:43,  4.16s/it]

Loss: 0.7034


...

[Epoch 10] Training: 100%|█████████▉| 471/473 [10:22<00:02,  1.32s/it]

Loss: 0.0009


[Epoch 10] Training: 100%|█████████▉| 472/473 [10:23<00:01,  1.32s/it]

Loss: 0.0003


[Teacher] Epoch 10 | Train Loss: 0.0035 | Val Acc: 0.9805 | Val AUC: 0.9986 | Time: 689.47s

Total training time: 6896.58s
Average time per epoch: 689.66s


In [ ]:
from sklearn.metrics import classification_report

def evaluate_teacher(model, dataloader, device='cuda'):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for x, y in dataloader:
            x = x.to(device)
            y = y.cpu().numpy()
            outputs = model(x)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(y)

    print("\n=== Classification Report ===")
    print(classification_report(all_labels, all_preds, digits=4))

In [ ]:
evaluate_teacher(teacher, test_loader)


=== Classification Report ===
              precision    recall  f1-score   support

           0     0.9762    0.9885    0.9823     20000
           1     0.9884    0.9758    0.9821     20000

    accuracy                         0.9822     40000
   macro avg     0.9823    0.9822    0.9822     40000
weighted avg     0.9823    0.9822    0.9822     40000

